# RAG Pipeline Demo

This notebook walks through loading, chunking, embedding, retrieval, prompt construction, and answer generation.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from utils import settings, format_source
settings

## 1. Load PDF pages

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

documents = []
for path in sorted(settings.sample_data_dir.glob('*.pdf')):
    pages = PyPDFLoader(str(path)).load()
    for page in pages:
        page.metadata['page'] = int(page.metadata.get('page', 0)) + 1
        page.metadata['source'] = path.name
    documents.extend(pages)

len(documents), documents[0].metadata

## 2. Split pages into chunks

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=settings.chunk_size,
    chunk_overlap=settings.chunk_overlap,
)
chunks = splitter.split_documents(documents)
len(chunks)

In [ ]:
print(chunks[0].page_content)
print(chunks[0].metadata)

## 3. Generate an example embedding

In [ ]:
from langchain_ollama import OllamaEmbeddings

embedding_model = OllamaEmbeddings(
    model=settings.embedding_model,
    base_url=settings.ollama_base_url,
)
vector = embedding_model.embed_query('How many leave days are provided?')
len(vector), vector[:10]

## 4. Store chunks in ChromaDB

In [ ]:
from langchain_chroma import Chroma

vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    collection_name='rag-notebook-demo',
)
vector_store._collection.count()

## 5. Retrieve relevant chunks

In [ ]:
question = 'How many annual leave days do employees receive?'
results = vector_store.similarity_search_with_score(question, k=4)

for doc, score in results:
    print(format_source(doc), score)
    print(doc.page_content)
    print('-' * 80)

## 6. Build context and prompt

In [ ]:
retrieved_docs = [doc for doc, _ in results]
context = '\n\n'.join(
    f'[Source: {format_source(doc)}]\n{doc.page_content}'
    for doc in retrieved_docs
)

prompt = f'''You are a document question-answering assistant.
Use only the supplied context. If the answer is not present, say that
the provided documents do not contain enough information.

Context:
{context}

Question:
{question}
'''
print(prompt)

## 7. Generate the answer

In [ ]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model=settings.chat_model,
    base_url=settings.ollama_base_url,
    temperature=0.1,
)
response = llm.invoke(prompt)
print(response.content)